# TF-IDF + Logistic Regression Baseline

Classical baseline for spoiler detection on IMDb and Goodreads.

Experiments:
1. IMDb → IMDb
2. Goodreads → Goodreads
3. IMDb → Goodreads

The validation sets are used only for selecting the Logistic Regression regularization parameter `C`.
The test sets are evaluated only after model selection.


In [1]:
!git clone -b dev https://github.com/rafaelTamm/NLE-Project-Spoiler-Detection.git
%cd NLE-Project-Spoiler-Detection


Cloning into 'NLE-Project-Spoiler-Detection'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (66/66), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 66 (delta 26), reused 16 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (66/66), 31.45 MiB | 10.86 MiB/s, done.
Resolving deltas: 100% (26/26), done.
Updating files: 100% (18/18), done.
/content/NLE-Project-Spoiler-Detection


In [2]:
!git branch


* dev


## Imports


In [3]:
import os
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


## Load data


In [4]:
imdb_train = pd.read_csv("data/imdb_train.csv")
imdb_val = pd.read_csv("data/imdb_val.csv")
imdb_test = pd.read_csv("data/imdb_test.csv")

goodreads_train = pd.read_csv("data/goodreads_train.csv")
goodreads_val = pd.read_csv("data/goodreads_val.csv")
goodreads_test = pd.read_csv("data/goodreads_test.csv")


In [5]:
print("IMDb")
print("Train:", imdb_train.shape)
print("Validation:", imdb_val.shape)
print("Test:", imdb_test.shape)

print("\nGoodreads")
print("Train:", goodreads_train.shape)
print("Validation:", goodreads_val.shape)
print("Test:", goodreads_test.shape)


IMDb
Train: (13913, 3)
Validation: (2821, 3)
Test: (3266, 3)

Goodreads
Train: (13982, 3)
Validation: (2934, 3)
Test: (3084, 3)


In [6]:
print("IMDb Train:")
print(imdb_train["is_spoiler"].value_counts())

print("\nGoodreads Train:")
print(goodreads_train["is_spoiler"].value_counts())


IMDb Train:
is_spoiler
False    6983
True     6930
Name: count, dtype: int64

Goodreads Train:
is_spoiler
True     7000
False    6982
Name: count, dtype: int64


## Evaluation helper

The positive-class metrics refer to `is_spoiler=True`.
Macro F1 gives equal weight to both classes and is used as the main metric for model selection.


In [7]:
def evaluate_model(y_true, y_pred, experiment_name):
    results = {
        "Experiment": experiment_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0)
    }

    print(experiment_name)
    print(f"Accuracy:  {results['Accuracy']:.4f}")
    print(f"Precision: {results['Precision']:.4f}")
    print(f"Recall:    {results['Recall']:.4f}")
    print(f"F1:        {results['F1']:.4f}")
    print(f"Macro F1:  {results['Macro F1']:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    return results


## Experiment 1: IMDb → IMDb


In [8]:
tfidf_imdb = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 1),
    min_df=2
)

X_imdb_train = tfidf_imdb.fit_transform(imdb_train["review_text"])
X_imdb_val = tfidf_imdb.transform(imdb_val["review_text"])
X_imdb_test = tfidf_imdb.transform(imdb_test["review_text"])

y_imdb_train = imdb_train["is_spoiler"]
y_imdb_val = imdb_val["is_spoiler"]
y_imdb_test = imdb_test["is_spoiler"]

print("Train:", X_imdb_train.shape)
print("Validation:", X_imdb_val.shape)
print("Test:", X_imdb_test.shape)


Train: (13913, 10000)
Validation: (2821, 10000)
Test: (3266, 10000)


In [9]:
best_f1_imdb = -1
best_C_imdb = None

for C in [0.1, 1, 10]:
    model = LogisticRegression(
        C=C,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_imdb_train, y_imdb_train)
    val_predictions = model.predict(X_imdb_val)

    val_f1 = f1_score(
        y_imdb_val,
        val_predictions,
        average="macro",
        zero_division=0
    )

    print(f"C={C}: Validation Macro F1 = {val_f1:.4f}")

    if val_f1 > best_f1_imdb:
        best_f1_imdb = val_f1
        best_C_imdb = C

print(f"\nBest C for IMDb: {best_C_imdb}")
print(f"Best IMDb Validation Macro F1: {best_f1_imdb:.4f}")


C=0.1: Validation Macro F1 = 0.6573
C=1: Validation Macro F1 = 0.6664
C=10: Validation Macro F1 = 0.6554

Best C for IMDb: 1
Best IMDb Validation Macro F1: 0.6664


In [10]:
logreg_imdb = LogisticRegression(
    C=best_C_imdb,
    max_iter=1000,
    random_state=42
)

logreg_imdb.fit(X_imdb_train, y_imdb_train)

imdb_predictions = logreg_imdb.predict(X_imdb_test)

result_imdb = evaluate_model(
    y_imdb_test,
    imdb_predictions,
    "IMDb → IMDb"
)


IMDb → IMDb
Accuracy:  0.6846
Precision: 0.6864
Recall:    0.6669
F1:        0.6765
Macro F1:  0.6844

Confusion Matrix:
[[1159  492]
 [ 538 1077]]

Classification Report:
              precision    recall  f1-score   support

       False       0.68      0.70      0.69      1651
        True       0.69      0.67      0.68      1615

    accuracy                           0.68      3266
   macro avg       0.68      0.68      0.68      3266
weighted avg       0.68      0.68      0.68      3266



## Experiment 2: Goodreads → Goodreads


In [11]:
tfidf_goodreads = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 1),
    min_df=2
)

X_goodreads_train = tfidf_goodreads.fit_transform(goodreads_train["review_text"])
X_goodreads_val = tfidf_goodreads.transform(goodreads_val["review_text"])
X_goodreads_test = tfidf_goodreads.transform(goodreads_test["review_text"])

y_goodreads_train = goodreads_train["is_spoiler"]
y_goodreads_val = goodreads_val["is_spoiler"]
y_goodreads_test = goodreads_test["is_spoiler"]

print("Train:", X_goodreads_train.shape)
print("Validation:", X_goodreads_val.shape)
print("Test:", X_goodreads_test.shape)


Train: (13982, 10000)
Validation: (2934, 10000)
Test: (3084, 10000)


In [12]:
best_f1_goodreads = -1
best_C_goodreads = None

for C in [0.1, 1, 10]:
    model = LogisticRegression(
        C=C,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_goodreads_train, y_goodreads_train)
    val_predictions = model.predict(X_goodreads_val)

    val_f1 = f1_score(
        y_goodreads_val,
        val_predictions,
        average="macro",
        zero_division=0
    )

    print(f"C={C}: Validation Macro F1 = {val_f1:.4f}")

    if val_f1 > best_f1_goodreads:
        best_f1_goodreads = val_f1
        best_C_goodreads = C

print(f"\nBest C for Goodreads: {best_C_goodreads}")
print(f"Best Goodreads Validation Macro F1: {best_f1_goodreads:.4f}")


C=0.1: Validation Macro F1 = 0.7267
C=1: Validation Macro F1 = 0.7486
C=10: Validation Macro F1 = 0.7355

Best C for Goodreads: 1
Best Goodreads Validation Macro F1: 0.7486


In [13]:
logreg_goodreads = LogisticRegression(
    C=best_C_goodreads,
    max_iter=1000,
    random_state=42
)

logreg_goodreads.fit(X_goodreads_train, y_goodreads_train)

goodreads_predictions = logreg_goodreads.predict(X_goodreads_test)

result_goodreads = evaluate_model(
    y_goodreads_test,
    goodreads_predictions,
    "Goodreads → Goodreads"
)


Goodreads → Goodreads
Accuracy:  0.7244
Precision: 0.7172
Recall:    0.7386
F1:        0.7277
Macro F1:  0.7243

Confusion Matrix:
[[1098  448]
 [ 402 1136]]

Classification Report:
              precision    recall  f1-score   support

       False       0.73      0.71      0.72      1546
        True       0.72      0.74      0.73      1538

    accuracy                           0.72      3084
   macro avg       0.72      0.72      0.72      3084
weighted avg       0.72      0.72      0.72      3084



## Experiment 3: IMDb → Goodreads

The IMDb TF-IDF vectorizer and IMDb-trained Logistic Regression model are reused without fitting on Goodreads.


In [14]:
X_goodreads_cross = tfidf_imdb.transform(
    goodreads_test["review_text"]
)

cross_predictions = logreg_imdb.predict(
    X_goodreads_cross
)

result_cross = evaluate_model(
    y_goodreads_test,
    cross_predictions,
    "IMDb → Goodreads"
)


IMDb → Goodreads
Accuracy:  0.6962
Precision: 0.6537
Recall:    0.8309
F1:        0.7317
Macro F1:  0.6907

Confusion Matrix:
[[ 869  677]
 [ 260 1278]]

Classification Report:
              precision    recall  f1-score   support

       False       0.77      0.56      0.65      1546
        True       0.65      0.83      0.73      1538

    accuracy                           0.70      3084
   macro avg       0.71      0.70      0.69      3084
weighted avg       0.71      0.70      0.69      3084



## Experiment 4: Goodreads → IMDb

For the reverse cross-domain experiment, the Goodreads TF-IDF vectorizer and Goodreads-trained Logistic Regression model are reused without fitting on IMDb.


In [15]:
X_imdb_cross = tfidf_goodreads.transform(
    imdb_test["review_text"]
)

reverse_cross_predictions = logreg_goodreads.predict(
    X_imdb_cross
)

result_reverse_cross = evaluate_model(
    y_imdb_test,
    reverse_cross_predictions,
    "Goodreads → IMDb"
)


Goodreads → IMDb
Accuracy:  0.6586
Precision: 0.6539
Recall:    0.6576
F1:        0.6558
Macro F1:  0.6586

Confusion Matrix:
[[1089  562]
 [ 553 1062]]

Classification Report:
              precision    recall  f1-score   support

       False       0.66      0.66      0.66      1651
        True       0.65      0.66      0.66      1615

    accuracy                           0.66      3266
   macro avg       0.66      0.66      0.66      3266
weighted avg       0.66      0.66      0.66      3266



## Majority baselines


In [16]:
dummy_imdb = DummyClassifier(
    strategy="most_frequent",
    random_state=42
)

dummy_imdb.fit(X_imdb_train, y_imdb_train)

result_dummy_imdb = evaluate_model(
    y_imdb_test,
    dummy_imdb.predict(X_imdb_test),
    "Majority → IMDb"
)

dummy_goodreads = DummyClassifier(
    strategy="most_frequent",
    random_state=42
)

dummy_goodreads.fit(X_goodreads_train, y_goodreads_train)

result_dummy_goodreads = evaluate_model(
    y_goodreads_test,
    dummy_goodreads.predict(X_goodreads_test),
    "Majority → Goodreads"
)


Majority → IMDb
Accuracy:  0.5055
Precision: 0.0000
Recall:    0.0000
F1:        0.0000
Macro F1:  0.3358

Confusion Matrix:
[[1651    0]
 [1615    0]]

Classification Report:
              precision    recall  f1-score   support

       False       0.51      1.00      0.67      1651
        True       0.00      0.00      0.00      1615

    accuracy                           0.51      3266
   macro avg       0.25      0.50      0.34      3266
weighted avg       0.26      0.51      0.34      3266

Majority → Goodreads
Accuracy:  0.4987
Precision: 0.4987
Recall:    1.0000
F1:        0.6655
Macro F1:  0.3328

Confusion Matrix:
[[   0 1546]
 [   0 1538]]

Classification Report:
              precision    recall  f1-score   support

       False       0.00      0.00      0.00      1546
        True       0.50      1.00      0.67      1538

    accuracy                           0.50      3084
   macro avg       0.25      0.50      0.33      3084
weighted avg       0.25      0.50      0.33 

## Compact cross-domain error analysis

This section extracts a small reproducible sample of false positives and false negatives from both cross-domain experiments. These examples can later be inspected manually for recurring patterns such as implicit spoilers, trigger words, domain-specific vocabulary, or missing context.


In [17]:
def collect_errors(df, y_true, y_pred, experiment_name, n_per_type=10, random_state=42):
    errors = df[["review_text", "is_spoiler"]].copy()
    errors["prediction"] = y_pred
    errors["experiment"] = experiment_name

    false_positives = errors[
        (errors["is_spoiler"] == False) &
        (errors["prediction"] == True)
    ]

    false_negatives = errors[
        (errors["is_spoiler"] == True) &
        (errors["prediction"] == False)
    ]

    fp_sample = false_positives.sample(
        n=min(n_per_type, len(false_positives)),
        random_state=random_state
    ).copy()
    fp_sample["error_type"] = "false_positive"

    fn_sample = false_negatives.sample(
        n=min(n_per_type, len(false_negatives)),
        random_state=random_state
    ).copy()
    fn_sample["error_type"] = "false_negative"

    return pd.concat([fp_sample, fn_sample], ignore_index=True)


errors_imdb_to_goodreads = collect_errors(
    goodreads_test,
    y_goodreads_test,
    cross_predictions,
    "IMDb → Goodreads"
)

errors_goodreads_to_imdb = collect_errors(
    imdb_test,
    y_imdb_test,
    reverse_cross_predictions,
    "Goodreads → IMDb"
)

cross_domain_errors = pd.concat(
    [errors_imdb_to_goodreads, errors_goodreads_to_imdb],
    ignore_index=True
)

cross_domain_errors.head()


,review_text,is_spoiler,prediction,experiment,error_type
0,I was pulled in from the first page. Knowing i...,False,True,IMDb → Goodreads,false_positive
1,I love the Signal Bend series. Susan Fanetti h...,False,True,IMDb → Goodreads,false_positive
2,"Riley and her friend, Jen, are out at a club. ...",False,True,IMDb → Goodreads,false_positive
3,"We takes the form of the diary of D-503, a mat...",False,True,IMDb → Goodreads,false_positive
4,"Naw, my first three star book for the year! D:...",False,True,IMDb → Goodreads,false_positive


In [18]:
os.makedirs("results", exist_ok=True)

cross_domain_errors.to_csv(
    "results/cross_domain_error_sample.csv",
    index=False
)

print("Saved error sample to results/cross_domain_error_sample.csv")
print(cross_domain_errors["experiment"].value_counts())
print(cross_domain_errors["error_type"].value_counts())


Saved error sample to results/cross_domain_error_sample.csv
experiment
IMDb → Goodreads    20
Goodreads → IMDb    20
Name: count, dtype: int64
error_type
false_positive    20
false_negative    20
Name: count, dtype: int64


## Results


In [19]:
results_df = pd.DataFrame([
    result_imdb,
    result_goodreads,
    result_cross,
    result_reverse_cross,
    result_dummy_imdb,
    result_dummy_goodreads
])

results_df


,Experiment,Accuracy,Precision,Recall,F1,Macro F1
0,IMDb → IMDb,0.684630,0.686424,0.666873,0.676508,0.684431
1,Goodreads → Goodreads,0.724384,0.717172,0.738622,0.727739,0.724342
2,IMDb → Goodreads,0.696174,0.653708,0.830949,0.731749,0.690734
3,Goodreads → IMDb,0.658604,0.653941,0.657585,0.655758,0.658580
4,Majority → IMDb,0.505511,0.000000,0.000000,0.000000,0.335774
5,Majority → Goodreads,0.498703,0.498703,1.000000,0.665513,0.332756


In [20]:
os.makedirs("results", exist_ok=True)

results_df.to_csv(
    "results/baseline_results.csv",
    index=False
)

print("Saved results to results/baseline_results.csv")


Saved results to results/baseline_results.csv
